In [24]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
import json

# ==========================================
# 1. LLM SETUP
# ==========================================
local_llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio-local", 
    model="local-model",
    temperature=0.3 # Low temp for logical planning
)

# ==========================================
# 2. THE WORKERS (Tools / Agents)
# ==========================================
def tool_fetch_patient_profile():
    """Worker 1: Gets patient data"""
    return json.dumps({"allergies": ["lactose"], "goal": "fitness", "calories": 1800})

def tool_analyze_food(food_item):
    """Worker 2: Gets food data"""
    db = {
        "paneer": {"lactose": "high", "protein": "18g"},
        "tofu": {"lactose": "none", "protein": "15g"}
    }
    return json.dumps(db.get(food_item.lower(), {"lactose": "unknown", "protein": "0g"}))

def tool_generate_plan(constraints):
    """Worker 3: The Chef that makes the food"""
    prompt = PromptTemplate.from_template("""
    You are a chef. Create a 1-day meal plan based on these constraints.
    Constraints: {constraints}
    Meal Plan:
    """)
    return local_llm.invoke(prompt.format(constraints=constraints)).content

# Map tool names to actual python functions
TOOLS = {
    "fetch_patient_profile": tool_fetch_patient_profile,
    "analyze_food": tool_analyze_food,
    "generate_meal_plan": tool_generate_plan
}

# ==========================================
# 3. THE PLANNER (The Manager)
# ==========================================
planner_prompt = PromptTemplate.from_template("""
You are an AI Planner. Your job is to break down a user request into a step-by-step plan using specific tools.

Available Tools:
- fetch_patient_profile: Use this to get user health data. Takes no arguments.
- analyze_food: Use this to check food safety/nutrition. Takes argument: {{"food": "food_name"}}
- generate_meal_plan: Use this ONLY at the end to make the actual food. Takes argument: {{"constraints": "summary of what to cook"}}

User Request: {request}

Respond ONLY in valid JSON format like this example:
[
  {{"step": 1, "tool": "fetch_patient_profile", "args": {{}}}},
  {{"step": 2, "tool": "analyze_food", "args": {{"food": "paneer"}}}},
  {{"step": 3, "tool": "generate_meal_plan", "args": {{"constraints": "Fitness plan, 1800 cal, no lactose, substitute paneer with tofu"}}}}
]

JSON Plan:
""")

def normalize_args(tool_name, args):
    if tool_name == "analyze_food":
        # Accept multiple planner variations
        food_value = args.get("food") or args.get("item") or args.get("name")
        if food_value:
            return {"food_item": food_value}  # ✅ matches tool_analyze_food
        return {}
    return args


def run_planner_system(user_query: str):
    print("🧠 [Planner] Analyzing request and creating a plan...")
    
    # Step 1: Ask LLM to create plan
    plan_response = local_llm.invoke(planner_prompt.format(request=user_query)).content
    print("response plan=========>", plan_response)
    
    if "```json" in plan_response:
        plan_response = plan_response.split("```json")[1].split("```")[0]
    
    try:
        plan_steps = json.loads(plan_response)
    except json.JSONDecodeError:
        return "Error: Planner failed to create a valid plan."
    
    print(f"📋 [Planner] Created {len(plan_steps)} step plan:")
    
    context_memory = {}
    
    # Step 2: Execute plan step-by-step
    for step in plan_steps:
        tool_name = step.get("tool")
        raw_args = step.get("args", {})
        
        print(f"   -> Executing Step {step['step']}: {tool_name} with args {raw_args}")
        
        tool_function = TOOLS.get(tool_name)
        if not tool_function:
            print(f"   -> [Error] Tool {tool_name} not found!")
            context_memory[tool_name] = {"error": "Tool not found"}
            continue
        
        # Normalize args to match function signature
        args = normalize_args(tool_name, raw_args)
        print(f"   -> Normalized args: {args}")
        
        try:
            result = tool_function(**args)
            context_memory[tool_name] = result
            print(f"   -> [Result] Success. Saved to memory.")
        
        except TypeError as ex:
            print(f"error----> Argument mismatch: {str(ex)}")
            context_memory[tool_name] = {"error": f"Argument mismatch: {str(ex)}"}
            # Optional: stop execution if a critical step fails
            # break
        
        except Exception as ex:
            print(f"error----> {str(ex)}")
            context_memory[tool_name] = {"error": str(ex)}
    
    # Step 3: Return final output
    final_output = context_memory.get(
        "generate_meal_plan",
        "System error: Meal plan was not generated."
    )
    
    return final_output



In [25]:
# ==========================================
# 5. SIMPLE INPUT
# ==========================================
# if __name__ == "__main__":
print("=== Planner-Based Nutrition System ===")
print("Type 'quit' to exit.\n")
user_input = "Can you plan a good diet for me for more lactose"
# while True:
#     user_input = input("You: ")
#     if user_input.lower() == 'quit':
#         break
        
result = run_planner_system(user_input)
print("\n🤖 Final Output:\n" + result + "\n" + "-"*50 + "\n")

=== Planner-Based Nutrition System ===
Type 'quit' to exit.

🧠 [Planner] Analyzing request and creating a plan...
response plan=========> [
  {"step": 1, "tool": "fetch_patient_profile", "args": {}},
  {"step": 2, "tool": "analyze_food", "args": {"food": "tofu"}},
  {"step": 3, "tool": "generate_meal_plan", "args": {"constraints": "Fitness plan, 1800 cal, no lactose, substitute tofu with paneer"}}
]
📋 [Planner] Created 3 step plan:
   -> Executing Step 1: fetch_patient_profile with args {}
   -> Normalized args: {}
   -> [Result] Success. Saved to memory.
   -> Executing Step 2: analyze_food with args {'food': 'tofu'}
   -> Normalized args: {'food_item': 'tofu'}
   -> [Result] Success. Saved to memory.
   -> Executing Step 3: generate_meal_plan with args {'constraints': 'Fitness plan, 1800 cal, no lactose, substitute tofu with paneer'}
   -> Normalized args: {'constraints': 'Fitness plan, 1800 cal, no lactose, substitute tofu with paneer'}
   -> [Result] Success. Saved to memory.

🤖 Fi

In [26]:

# ==========================================
# 2. THE WORKERS (Tools / Agents)
# ==========================================
def tool_fetch_patient_profile():
    return json.dumps({"allergies": ["lactose"], "goal": "fitness", "calories": 1800})

# FIX 1: Changed 'food_item' to 'food' to match the LLM's JSON output
def tool_analyze_food(food): 
    db = {
        "paneer": {"lactose": "high", "protein": "18g", "category": "dairy"},
        "tofu": {"lactose": "none", "protein": "15g", "category": "soy"},
        "greek_yogurt": {"lactose": "high", "category": "dairy"} # Added to catch the mistake!
    }
    return json.dumps(db.get(food.lower(), {"lactose": "unknown", "protein": "0g"}))

def tool_generate_plan(context_data):
    """Worker 3: Now takes ALL previous context to make smart decisions"""
    prompt = PromptTemplate.from_template("""
    You are a strict chef. Create a 1-day meal plan based on this data.
    
    DATA FROM PREVIOUS STEPS:
    {context_data}
    
    RULES:
    1. DO NOT guess. Use the data provided.
    2. If a food has "lactose: high", DO NOT include it. Find a substitute.
    3. If you don't have data on a food, assume it is unsafe.
    
    Meal Plan:
    """)
    return local_llm.invoke(prompt.format(context_data=context_data)).content

TOOLS = {
    "fetch_patient_profile": tool_fetch_patient_profile,
    "analyze_food": tool_analyze_food,
    "generate_meal_plan": tool_generate_plan
}

# ==========================================
# 3. THE PLANNER (Upgraded to pass memory)
# ==========================================
planner_prompt = PromptTemplate.from_template("""
You are an AI Planner. Break down the request into steps.

Available Tools:
- fetch_patient_profile: Takes no arguments: {{}}
- analyze_food: Takes: {{"food": "name"}}
- generate_meal_plan: Takes: {{"context_data": "PASTE ALL RESULTS FROM STEP 1 AND 2 HERE"}}

User Request: {request}

Respond ONLY in valid JSON:
[
  {{"step": 1, "tool": "fetch_patient_profile", "args": {{}}}},
  {{"step": 2, "tool": "analyze_food", "args": {{"food": "paneer"}}}},
  {{"step": 3, "tool": "generate_meal_plan", "args": {{"context_data": "Combine results from step 1 and 2 here"}}}}
]

JSON Plan:
""")

# ==========================================
# 4. THE EXECUTION ENGINE
# ==========================================
def run_planner_system(user_query: str):
    print("🧠 [Planner] Analyzing request and creating a plan...")
    
    plan_response = local_llm.invoke(planner_prompt.format(request=user_query)).content
    
    if "```json" in plan_response:
        plan_response = plan_response.split("```json")[1].split("```")[0]
        
    try:
        plan_steps = json.loads(plan_response)
    except json.JSONDecodeError:
        return "Error: Planner failed to create a valid plan."

    print(f"📋 [Planner] Created {len(plan_steps)} step plan:\n")
    
    context_memory = {}

    # FIX 2: Build a running string of all results gathered so far
    gathered_intel = ""

    for step in plan_steps:
        tool_name = step.get("tool")
        args = step.get("args", {})
        
        print(f"   -> Executing Step {step['step']}: {tool_name}")
        
        tool_function = TOOLS.get(tool_name)
        if not tool_function:
            continue
            
        # Execute tool
        result = tool_function(**args)
        
        # Save raw result
        context_memory[tool_name] = result
        
        # Add to our running intelligence report
        gathered_intel += f"- Result of {tool_name}: {result}\n"
        print(f"   -> [Result] Success. Data added to intel.\n")

    # Update the final step's arguments with the ACTUAL data gathered
    if "generate_meal_plan" in context_memory:
        # We override whatever the LLM guessed in step 3 with REAL data
        final_result = tool_generate_plan(context_data=gathered_intel)
        return final_result
    else:
        return "Error: Meal plan tool was not called."

# # ==========================================
# # 5. SIMPLE INPUT
# # ==========================================
# if __name__ == "__main__":
#     print("=== Planner-Based Nutrition System ===")
#     print("Type 'quit' to exit.\n")
    
#     while True:
#         user_input = input("You: ")
#         if user_input.lower() == 'quit':
#             break
            
#         result = run_planner_system(user_input)
#         print("\n🤖 Final Output:\n" + result + "\n" + "-"*50 + "\n")

In [27]:
# ==========================================
# 5. SIMPLE INPUT
# ==========================================
# if __name__ == "__main__":
print("=== Planner-Based Nutrition System ===")
print("Type 'quit' to exit.\n")
user_input = "Can you plan a good diet for me for more lactose"
# while True:
#     user_input = input("You: ")
#     if user_input.lower() == 'quit':
#         break
        
result = run_planner_system(user_input)
print("\n🤖 Final Output:\n" + result + "\n" + "-"*50 + "\n")

=== Planner-Based Nutrition System ===
Type 'quit' to exit.

🧠 [Planner] Analyzing request and creating a plan...
📋 [Planner] Created 3 step plan:

   -> Executing Step 1: fetch_patient_profile
   -> [Result] Success. Data added to intel.

   -> Executing Step 2: analyze_food
   -> [Result] Success. Data added to intel.

   -> Executing Step 3: generate_meal_plan
   -> [Result] Success. Data added to intel.


🤖 Final Output:
Breakfast:
- Oatmeal with fresh berries and a drizzle of honey (lactose: low)
- A glass of orange juice (lactose: low)

Mid-Morning Snack:
- A handful of almonds (lactose: low)

Lunch:
- Grilled chicken salad with mixed greens, cherry tomatoes, cucumber, and a balsamic vinaigrette dressing (lactose: low)

Afternoon Snack:
- A piece of fruit like an apple or banana (lactose: low)

Dinner:
- Baked salmon with a side of steamed broccoli and quinoa (lactose: low)
--------------------------------------------------

